# UNSW-NB15: Feature Engineering

In [223]:
import os, pickle, gc  # files, saving objects, freeing memory
import numpy as np  # for array
import pandas as pd  # for csv files and dataframe
import matplotlib.pyplot as plt  # for plotting
import seaborn as sns  # plotting
from scipy import stats

In [224]:
# Refuse stale full-dataset CSVs created before the raw-row cap was introduced.
with open('./final_ipynb', 'rb') as metadata_file:
    sample_metadata = pickle.load(metadata_file)
if 'raw_sampled_rows' not in sample_metadata:
    raise RuntimeError("Run 'About Dataset and It\'s Cleaning.ipynb' to create the capped raw sample first.")

# Loading the sampled data from disk
train = pd.read_csv('../data/train_alldata_EDA.csv')
test = pd.read_csv('../data/test_alldata_EDA.csv')
if len(train) + len(test) != sample_metadata['raw_sampled_rows']:
    raise RuntimeError("Prepared CSV row counts do not match the saved raw sample. Rerun the cleaning notebook.")

C:\Users\ENPEI\AppData\Local\Temp\ipykernel_64856\2786933953.py:8: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('../data/train_alldata_EDA.csv')


In [225]:
train.shape , test.shape

((35000, 49), (15000, 49))

In [226]:
# Utility function
def multi_corr(col1, col2="label", df=train):
    '''
    This function returns correlation between 2 given features.
    Also gives corr of the given features with "label" afetr applying log1p to it.
    '''
    corr = df[[col1, col2]].corr().iloc[0,1]
    log_corr = df[col1].apply(np.log1p).corr(df[col2])

    print("Correlation : {}\nlog_Correlation: {}".format(corr, log_corr))

In [227]:
def corr(col1, col2="label", df=train):
    """
    This function returns correlation between 2 given features
    """
    return df[[col1, col2]].corr().iloc[0,1]

## Removing highly correlated features

In [228]:
corr_matrix = train.corr(numeric_only=True).abs()

# Select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(np.bool))

# Find index of feature columns with correlation greater than 0.9
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]

In [229]:
# We don't want to use these features for plotting because these are having high corr
# And most likely have same kind of plots with already plotted feature
print(to_drop)

['sloss', 'dloss', 'dpkts', 'dwin', 'ltime', 'synack', 'ct_srv_dst', 'ct_src_dport_ltm', 'ct_dst_src_ltm']


In [230]:
# Reuse the dictionary saved by the first notebook (it already holds 'columns', 'binary_col', ...)
old_file = './final_ipynb'

if os.path.isfile(old_file):
    saved_dict = pickle.load(open(old_file, 'rb'))
else:
    saved_dict = {}
    print("WARNING: ./final_ipynb not found, starting with an empty saved_dict")

print(list(saved_dict.keys()))   # should contain 'columns' and 'binary_col'
saved_dict['corr_col'] = to_drop  # add the new key on top of the old ones

['raw_sampled_rows', 'raw_sample_limit', 'raw_sample_seed', 'raw_valid_rows_seen', 'columns', 'binary_col']


In [231]:
# removing the features from train and test data
train.drop(columns=to_drop, inplace=True)

In [232]:
test.drop(columns=to_drop, inplace=True)

In [233]:
train.shape, test.shape

((35000, 40), (15000, 40))

## Adding New Features

Refer: https://www.elastic.co/guide/en/ecs/master/ecs-network.html
* Network bytes: Total bytes trasferred by the network. It is sum of 'sbytes' (Source to destination bytes) and 'dbytes' (Destination to source bytes).

In [234]:
# creating new features
train['network_bytes'] = train['sbytes'] + train['dbytes']

In [235]:
train.shape, test.shape

((35000, 41), (15000, 40))

In [236]:
# Dropping columns which are not useful for the classification
# attack_cat is for multiclass classification
# all the other columns are address related and not present in sample train data
train.drop(['srcip', 'sport', 'dstip', 'dsport', 'attack_cat'], axis=1, inplace=True)

In [237]:
# To use during test data transformation
saved_dict['to_drop'] = ['srcip', 'sport', 'dstip', 'dsport', 'attack_cat']

In [238]:
train.shape, test.shape

((35000, 36), (15000, 40))

In [239]:
train["service"]=train["service"].fillna("others")


In [240]:
train["service"].value_counts()

service
others      17048
dns         10823
http         2884
ftp-data     1741
smtp         1122
ftp           711
ssh           640
pop3           19
snmp            6
dhcp            2
ssl             2
radius          1
irc             1
Name: count, dtype: int64

## Applying log1p on Numerical columns

During EDA we found that few numerical columns shows better visualization for pdf curves if we apply log1p to the columns.

So I thought to try log1p on all the columns and check the correlation value of the original column and log1p column with target column i.e. "label"

In [241]:
# Getting number of unique values of all the columns
# If the unique values are high that means it has continuous set of values
col_unique_values = train.nunique()
col_unique_values

proto                 114
state                   9
dur                 20861
sbytes               1660
dbytes               1956
sttl                   12
dttl                    7
service                13
sload               23368
dload               22847
spkts                 320
swin                    2
stcpb               20514
dtcpb               20509
smeansz               466
dmeansz               838
trans_depth             3
res_bdy_len           146
sjit                21296
djit                22230
stime               27529
sintpkt             21526
dintpkt             21273
tcprtt               2491
ackdat               1865
is_sm_ips_ports         2
ct_state_ttl            6
ct_flw_http_mthd       13
is_ftp_login            2
ct_ftp_cmd              8
ct_srv_src             63
ct_dst_ltm             61
ct_src_ltm             61
ct_dst_sport_ltm       49
label                   2
network_bytes        3208
dtype: int64

In [242]:
# If the unique values are getter than some threshould than we will check its corr
col = col_unique_values[col_unique_values>200].index

In [243]:
# Checking corr value of original col and log1p applied col
# Taking those columns whose unique values are getter than some threshould
for column in col:
    print("{:-^30}".format(column))
    multi_corr(column)

-------------dur--------------
Correlation : 0.005409777249415992
log_Correlation: -0.0320090475220922
------------sbytes------------
Correlation : 0.007323196999468554
log_Correlation: -0.34886669278554033
------------dbytes------------
Correlation : -0.08104616065634258
log_Correlation: -0.5150355185640383
------------sload-------------
Correlation : 0.18021087869457675
log_Correlation: 0.3415917114688575
------------dload-------------
Correlation : -0.2195404135434346
log_Correlation: -0.6003017059993698
------------spkts-------------
Correlation : -0.11774865569300763
log_Correlation: -0.3127556573225859
------------stcpb-------------
Correlation : -0.22852140006734376
log_Correlation: -0.30828562060350356
------------dtcpb-------------
Correlation : -0.2313391336814594
log_Correlation: -0.3084761138383791
-----------smeansz------------
Correlation : -0.05340442551602878
log_Correlation: -0.14055718324673963
-----------dmeansz------------
Correlation : -0.27104333596831404
log_Corr

In [244]:
# Will apply log1p on this columns and remove original columns
log1p_col = ['dur', 'sbytes', 'dbytes', 'sload', 'dload', 'spkts', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'sjit', 'djit', 'network_bytes']

In [245]:
saved_dict['log1p_col'] = log1p_col

In [246]:
# mode values of every features, will use to fill Null values of test
mode_dict = train.mode().iloc[0].to_dict()
mode_dict

{'proto': 'tcp',
 'state': 'FIN',
 'dur': 9e-06,
 'sbytes': 146.0,
 'dbytes': 0.0,
 'sttl': 31.0,
 'dttl': 29.0,
 'service': 'others',
 'sload': 50666664.0,
 'dload': 0.0,
 'spkts': 2.0,
 'swin': 255.0,
 'stcpb': 0.0,
 'dtcpb': 0.0,
 'smeansz': 132.0,
 'dmeansz': 0.0,
 'trans_depth': 0.0,
 'res_bdy_len': 0.0,
 'sjit': 0.0,
 'djit': 0.0,
 'stime': 1421927443,
 'sintpkt': 0.009,
 'dintpkt': 0.0,
 'tcprtt': 0.0,
 'ackdat': 0.0,
 'is_sm_ips_ports': 0.0,
 'ct_state_ttl': 0.0,
 'ct_flw_http_mthd': 0.0,
 'is_ftp_login': 0.0,
 'ct_ftp_cmd': 0.0,
 'ct_srv_src': 1.0,
 'ct_dst_ltm': 2.0,
 'ct_src_ltm': 2.0,
 'ct_dst_sport_ltm': 1.0,
 'label': 0.0,
 'network_bytes': 324.0}

In [247]:
def log1p_transform(col, df=train):
    '''
    Apply log1p on given column.
    Remove the original cola and keep log1p applied col
    '''
    new_col = col+'_log1p'
    df[new_col] = df[col].apply(np.log1p)
    df.drop(col, axis=1, inplace=True)

In [248]:
# Transforming columns with log1p
for col in log1p_col:
    log1p_transform(col, df=train)

In [249]:
train.shape

(35000, 36)

In [250]:
train.columns

Index(['proto', 'state', 'sttl', 'dttl', 'service', 'swin', 'trans_depth',
       'res_bdy_len', 'stime', 'sintpkt', 'dintpkt', 'tcprtt', 'ackdat',
       'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login',
       'ct_ftp_cmd', 'ct_srv_src', 'ct_dst_ltm', 'ct_src_ltm',
       'ct_dst_sport_ltm', 'label', 'dur_log1p', 'sbytes_log1p',
       'dbytes_log1p', 'sload_log1p', 'dload_log1p', 'spkts_log1p',
       'stcpb_log1p', 'dtcpb_log1p', 'smeansz_log1p', 'dmeansz_log1p',
       'sjit_log1p', 'djit_log1p', 'network_bytes_log1p'],
      dtype='object')

In [251]:
train.shape, test.shape

((35000, 36), (15000, 40))

In [252]:
# creating x and y set from the dataset
x_train, y_train = train.drop(columns=['label']), train['label']
x_test, y_test = test.drop(columns=['label']), test['label']

In [253]:
print(x_train.shape, y_train.shape)
print()
print(x_test.shape, y_test.shape)

(35000, 35) (35000,)

(15000, 39) (15000,)


In [254]:
# Final model-ready train/test pickles are saved after both splits are preprocessed.

In [255]:
# getting categorical and numerical columns in 2 diff lists
cat_col = ['proto', 'service', 'state']
num_col = [c for c in x_train.columns if c not in cat_col]  # keeps the column order

In [256]:
# To use later, during test data cleaning
saved_dict['cat_col'] = cat_col
saved_dict['num_col'] = num_col

In [257]:
x_train.head()

,proto,state,sttl,dttl,service,swin,trans_depth,res_bdy_len,stime,sintpkt,...,sload_log1p,dload_log1p,spkts_log1p,stcpb_log1p,dtcpb_log1p,smeansz_log1p,dmeansz_log1p,sjit_log1p,djit_log1p,network_bytes_log1p
0,udp,INT,60,0,dns,0,0,0,1424230633,0.004000,...,19.391460,0.000000,1.098612,0.000000,0.000000,4.890349,0.000000,0.000000,0.000000,5.579730
1,tcp,FIN,31,29,others,255,0,0,1424241711,65.385987,...,8.832997,12.578391,6.137727,21.879774,20.837237,4.043051,7.152269,10.137709,9.828533,13.929444
2,tcp,FIN,31,29,others,255,0,0,1424230578,0.290087,...,14.629481,14.741797,3.218876,21.524712,21.524846,4.465908,4.574711,2.854091,3.024893,8.382747
3,udp,CON,31,29,dns,0,0,0,1421971106,0.011000,...,13.120444,13.340506,1.098612,0.000000,0.000000,4.189655,4.406719,0.000000,0.000000,5.680173
4,tcp,FIN,31,29,others,255,0,0,1421933966,0.239677,...,15.173688,15.569136,3.496508,20.891615,20.891861,4.770685,5.375278,2.761722,2.828209,9.138522


## Outlier Detection

Extreme values (outliers) can distort ML models. However, in network intrusion data, **outliers are often the attacks themselves** (e.g., massive byte counts in a DDoS).

We detect them to understand the data, not to blindly delete them. 

### IQR Method (Interquartile Range)
Finds values outside `Q1 - 1.5 * IQR` and `Q3 + 1.5 * IQR`. Highly robust because it uses percentiles instead of the mean.

In [258]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Function to find outlier bounds using IQR
def get_iqr_bounds(df, columns):
    Q1 = df[columns].quantile(0.25)
    Q3 = df[columns].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return lower_bound, upper_bound

# 2. Get the bounds for your numerical columns
lower_bounds, upper_bounds = get_iqr_bounds(x_train, num_col)

# 3. Detect outliers
outliers_mask = (x_train[num_col] < lower_bounds) | (x_train[num_col] > upper_bounds)

# 4. Calculate the PROPORTION (percentage) instead of the count
# .mean() on a boolean mask gives the proportion of True values
outlier_proportions = outliers_mask.mean().sort_values(ascending=False) * 100

print("--- Proportion of Outliers per Feature (IQR Method) ---")
# Print only columns that have at least 1 outlier, formatted as percentages
valid_proportions = outlier_proportions[outlier_proportions > 0]
for col, prop in valid_proportions.items():
    print(f"{col:25s} : {prop:.2f}%")

# 5. Plot Boxplots (Diagrammes en moustaches)
# n_cols = 4
# n_rows = int(np.ceil(len(num_col) / n_cols))

# fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
# axes = axes.flatten()

# for i, col in enumerate(num_col):
#     sns.boxplot(y=x_train[col], ax=axes[i], color='skyblue')
#     axes[i].set_title(col)
#     axes[i].set_ylabel('')

# for j in range(i + 1, len(axes)):
#     fig.delaxes(axes[j])

# plt.tight_layout()
# plt.show()

--- Proportion of Outliers per Feature (IQR Method) ---
dttl                      : 23.72%
sttl                      : 23.66%
ct_dst_sport_ltm          : 22.38%
dload_log1p               : 19.65%
dmeansz_log1p             : 19.63%
dintpkt                   : 17.92%
sintpkt                   : 17.40%
dur_log1p                 : 16.91%
ct_dst_ltm                : 14.41%
ct_state_ttl              : 14.28%
ct_src_ltm                : 13.43%
ct_srv_src                : 12.53%
sload_log1p               : 10.90%
trans_depth               : 8.10%
ct_flw_http_mthd          : 8.10%
res_bdy_len               : 7.31%
smeansz_log1p             : 5.81%
ackdat                    : 4.98%
tcprtt                    : 4.56%
is_ftp_login              : 1.74%
ct_ftp_cmd                : 1.74%
is_sm_ips_ports           : 0.18%
sbytes_log1p              : 0.05%
djit_log1p                : 0.02%
spkts_log1p               : 0.00%


## Interpreting the Outlier Proportions

The IQR results show high outlier rates for `dttl` (24%) and `sttl` (24%), but 0% for `dbytes_log1p`. This is mathematically correct but misleading for network data:

1. **Discrete Data:** Features like `dttl` are mostly 29. The IQR becomes tiny, flagging normal high values (like 252) as outliers.
2. **Log1p Effect:** The log transformation compressed extreme byte counts, hiding them from the IQR math.
3. **Binary Columns:** Columns like `is_ftp_login` are just 0s and 1s, confusing the IQR formula.

**Conclusion:** Do not drop these "outliers." In network traffic, outliers are often the actual attacks. Use `RobustScaler` or tree-based models (Random Forest/XGBoost) which handle them automatically.

### Z-Score Method
Finds values more than 3 standard deviations from the mean.

In [259]:
import pandas as pd
import numpy as np
from scipy import stats

# 1. Calculate Z-scores for all numerical columns
z_scores = np.abs(stats.zscore(x_train[num_col]))

# 2. Flag anything greater than 3 (99.7% of normal data falls within 3 std devs)
z_outliers_mask = z_scores > 3

# 3. Calculate the proportion (percentage) instead of count
z_outlier_proportions = pd.Series(z_outliers_mask.mean(axis=0), index=x_train[num_col].columns) * 100
z_outlier_proportions = z_outlier_proportions[z_outlier_proportions > 0].sort_values(ascending=False)

print("--- Proportion of Outliers per Feature (Z-Score > 3) ---")
for col, prop in z_outlier_proportions.items():
    print(f"{col:25s} : {prop:.2f}%")

--- Proportion of Outliers per Feature (Z-Score > 3) ---
trans_depth               : 8.10%
smeansz_log1p             : 5.51%
dttl                      : 3.48%
ct_src_ltm                : 2.96%
ct_dst_ltm                : 2.92%
ct_dst_sport_ltm          : 2.91%
dur_log1p                 : 2.44%
ct_srv_src                : 2.31%
tcprtt                    : 1.96%
ackdat                    : 1.92%
is_ftp_login              : 1.74%
ct_ftp_cmd                : 1.74%
res_bdy_len               : 0.82%
sload_log1p               : 0.63%
sintpkt                   : 0.60%
ct_flw_http_mthd          : 0.59%
ct_state_ttl              : 0.34%
dintpkt                   : 0.27%
is_sm_ips_ports           : 0.18%
sbytes_log1p              : 0.07%
sjit_log1p                : 0.03%
spkts_log1p               : 0.01%
djit_log1p                : 0.01%
network_bytes_log1p       : 0.00%


## Z-Score vs IQR Outlier Detection

**Comparison:**
- **IQR:** Flagged 23.64% of `dttl` as outliers. Too strict for discrete data; over-flags normal traffic.
- **Z-Score:** Flagged only 3.36% of `dttl`. Too lenient; extreme values inflate the mean/std, hiding true outliers.
- **Both fail** on `log1p` features (0% outliers) because transformation compressed extremes.

**Conclusion:**
Do not drop these outliers. Attacks are extremes. 
Use **Tree-based models** (immune to outliers) or **RobustScaler** (distance-based models).

## Standardizing
    
As we have seen that the range of few features in this dataset is very large. So we will keep everything within certain range by applying standardscaler. After this all the features will have mean 0 and std 1

In [260]:
# Standardizing the data
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
scaler = StandardScaler()
scaler = scaler.fit(x_train[num_col])

In [261]:
x_train[num_col] = scaler.transform(x_train[num_col])

In [262]:
x_train.head()

,proto,state,sttl,dttl,service,swin,trans_depth,res_bdy_len,stime,sintpkt,...,sload_log1p,dload_log1p,spkts_log1p,stcpb_log1p,dtcpb_log1p,smeansz_log1p,dmeansz_log1p,sjit_log1p,djit_log1p,network_bytes_log1p
0,udp,INT,-0.038105,-0.713187,dns,-1.195706,-0.293398,-0.085009,0.849227,-0.070672,...,1.831949,-1.883058,-1.079531,-1.189628,-1.189259,0.558501,-1.853154,-0.995272,-0.899453,-1.061084
1,tcp,FIN,-0.426176,-0.047006,others,0.836326,-0.293398,-0.085009,0.859002,-0.047314,...,-1.513505,0.331932,2.619599,0.902696,0.802625,-0.793369,1.242452,2.125905,2.579890,2.496634
2,tcp,FIN,-0.426176,-0.047006,others,0.836326,-0.293398,-0.085009,0.849179,-0.070570,...,0.323114,0.712897,0.476919,0.868742,0.868355,-0.118697,0.126847,-0.116560,0.171372,0.133250
3,udp,CON,-0.426176,-0.047006,dns,-1.195706,-0.293398,-0.085009,-1.144486,-0.070670,...,-0.155025,0.466137,-1.079531,-1.189628,-1.189259,-0.559462,0.054138,-0.995272,-0.899453,-1.018287
4,tcp,FIN,-0.426176,-0.047006,others,0.836326,-0.293398,-0.085009,-1.177257,-0.070588,...,0.495546,0.858587,0.680724,0.808200,0.807846,0.367576,0.473345,-0.144999,0.101745,0.455277


## Onehot Encoding

In our dataset we have few categorical columns with text data.
But ML models can't process text data it can process numbers.

So we have to convert categorical columns to numerical columns in some way.
We will use onehotencoder where we will assign 1 if the value is present for the row and rest of the columns will be 0.

In [263]:
from sklearn.preprocessing import OneHotEncoder
import numpy as np

# One-hot encoding
service_ = OneHotEncoder(dtype=np.int8,handle_unknown="ignore")
proto_ = OneHotEncoder(dtype=np.int8  ,handle_unknown="ignore")
state_ = OneHotEncoder(dtype=np.int8  ,handle_unknown="ignore")

ohe_service = service_.fit(x_train.service.values.reshape(-1, 1))
ohe_proto = proto_.fit(x_train.proto.values.reshape(-1, 1))
ohe_state = state_.fit(x_train.state.values.reshape(-1, 1))

In [264]:
pieces = []
for col, ohe in zip(['proto', 'service', 'state'], [ohe_proto, ohe_service, ohe_state]):
    x = ohe.transform(x_train[col].values.reshape(-1, 1)).astype(np.uint8)
    pieces.append(pd.DataFrame.sparse.from_spmatrix(
        x,
        columns=[col + '_' + str(i) for i in ohe.categories_[0]],
        index=x_train.index
    ))
    x_train.drop(columns=col, inplace=True)   # free the original column right away
    del x
    gc.collect()

x_train = pd.concat([x_train] + pieces, axis=1)   # a single concat at the end
del pieces
gc.collect()

0

In [265]:
x_train.head()

,sttl,dttl,swin,trans_depth,res_bdy_len,stime,sintpkt,dintpkt,tcprtt,ackdat,...,service_ssl,state_CLO,state_CON,state_ECO,state_FIN,state_INT,state_REQ,state_RST,state_URH,state_URN
0,-0.038105,-0.713187,-1.195706,-0.293398,-0.085009,0.849227,-0.070672,-0.056635,-0.148237,-0.144872,...,0,0,0,0,0,1,0,0,0,0
1,-0.426176,-0.047006,0.836326,-0.293398,-0.085009,0.859002,-0.047314,-0.031742,-0.126460,-0.123569,...,0,0,0,0,1,0,0,0,0,0
2,-0.426176,-0.047006,0.836326,-0.293398,-0.085009,0.849179,-0.070570,-0.056439,-0.130317,-0.137585,...,0,0,0,0,1,0,0,0,0,0
3,-0.426176,-0.047006,-1.195706,-0.293398,-0.085009,-1.144486,-0.070670,-0.056629,-0.148237,-0.144872,...,0,0,1,0,0,0,0,0,0,0
4,-0.426176,-0.047006,0.836326,-0.293398,-0.085009,-1.177257,-0.070588,-0.056449,-0.130892,-0.137535,...,0,0,0,0,1,0,0,0,0,0


Saving all the important parameters and objects to disk so that we can apply same process on test data

In [266]:
file_path = '.final_ipynb/'
os.makedirs(file_path, exist_ok=True)

In [267]:
pickle.dump(scaler, open(file_path+'scaler.pkl', 'wb'))  # Standard scaler

# Merge with the dictionary already on disk instead of replacing it:
# keys from this notebook refresh old values, keys that only exist on disk are kept
path = file_path + 'saved_dict.pkl'
if os.path.exists(path):
    with open(path, 'rb') as f:
        old_dict = pickle.load(f)
else:
    old_dict = {}

old_dict.update(saved_dict)
with open(path, 'wb') as f:
    pickle.dump(old_dict, f)  # Dictionary with important parameters

pickle.dump(mode_dict, open(file_path+'mode_dict.pkl', 'wb'))  # Dictionary with most frequent values of columns

In [268]:
# Onehot encoder for categorical columns
pickle.dump(ohe_proto, open(file_path+'ohe_proto.pkl', 'wb'))
pickle.dump(ohe_service, open(file_path+'ohe_service.pkl', 'wb'))
pickle.dump(ohe_state, open(file_path+'ohe_state.pkl', 'wb'))

In [269]:
# Final train/test pickles are saved after test preprocessing at the end of this notebook.

## Pipeline functions

We have to prepare a pipeline, where we can send raw data and get the output.

We will use test data to implement the pipeline. Here we will use all the parameters we have saved using train data.

Also standardize and onehot encode test data using train data objects for standardscaler and onehotencoder.

In [270]:
def clean_data(data):
    '''
    Cleans given raw data. Performs various cleaning, removes Null and wrong values.
    Check for columns datatype and fix them.
    '''
    numerical_col = data.select_dtypes(include=np.number).columns  # All the numerical columns list
    categorical_col = data.select_dtypes(exclude=np.number).columns  # All the categorical columns list
    
    # Cleaning the data
    for col in data.columns:
        val = mode_dict[col]  # Mode value of the column in train data
        data[col] = data[col].fillna(value=val)
        data[col] = data[col].replace(' ', value=val)
        data[col] = data[col].apply(lambda x:"None" if x=="-" else x)

        # Fixing binary columns
        if col in saved_dict['binary_col']:
            data[col] = np.where(data[col]>1, val, data[col])

    # Fixing datatype of columns
    bad_dtypes = list(set(categorical_col) - set(saved_dict['cat_col']))
    for bad_col in bad_dtypes:
        data[bad_col] = data[bad_col].astype(float)   # was data[col]: wrong column
    
    return data

In [271]:
def apply_log1p(data):
    '''
    Performs FE on the data. Apply log1p on the specified columns create new column and remove those original columns.
    '''
    for col in saved_dict['log1p_col']:
        new_col = col + '_log1p'  # New col name
        data[new_col] = data[col].apply(np.log1p)  # Creating new column on transformed data
        data.drop(col, axis=1, inplace=True)  # Removing old columns
    return data

In [272]:
def standardize(data):
    '''
    Stanardize the given data. Performs mean centering and varience scaling.
    Using stanardscaler object trained on train data.
    '''
    data[saved_dict['num_col']] = scaler.transform(data[saved_dict['num_col']])
    return data

In [273]:
def ohencoding(data):
    '''
    Onehot encoding the categorical columns.
    Add the ohe columns with the data and removes categorical columns.
    Using Onehotencoder objects trained on train data.
    '''
    pieces = []
    for col, ohe in [('proto', ohe_proto), ('service', ohe_service), ('state', ohe_state)]:
        X = ohe.transform(data[col].values.reshape(-1, 1)).astype(np.uint8)
        pieces.append(pd.DataFrame.sparse.from_spmatrix(
            X,
            columns=[col + '_' + str(c) for c in ohe.categories_[0]],
            index=data.index                      # same rows as data, avoids NaN rows
        ))

    data = data.drop(columns=['proto', 'service', 'state'])   # remove cat columns first
    return pd.concat([data] + pieces, axis=1)                 # a single concat

Loading all the objects from disk, that we have trained on train data.

In [274]:
# Parameters (the complete dictionary saved above)
saved_dict = pickle.load(open(file_path+'saved_dict.pkl', 'rb'))
# Mode value of all the columns
mode_dict = pickle.load(open(file_path+'mode_dict.pkl', 'rb'))
# Stanardscaler object
scaler = pickle.load(open(file_path+'scaler.pkl', 'rb'))

In [275]:
# One hot encoder objects
ohe_proto = pickle.load(open(file_path+'ohe_proto.pkl', 'rb'))
ohe_service = pickle.load(open(file_path+'ohe_service.pkl', 'rb'))
ohe_state = pickle.load(open(file_path+'ohe_state.pkl', 'rb'))

In [276]:
x_test.shape

(15000, 39)

In [277]:
# Resetting index of test data
x_test.reset_index(drop=True, inplace=True)

In [278]:
x_test.shape

(15000, 39)

In [279]:
x_test.columns

Index(['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes',
       'dbytes', 'sttl', 'dttl', 'service', 'sload', 'dload', 'spkts', 'swin',
       'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len',
       'sjit', 'djit', 'stime', 'sintpkt', 'dintpkt', 'tcprtt', 'ackdat',
       'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login',
       'ct_ftp_cmd', 'ct_srv_src', 'ct_dst_ltm', 'ct_src_ltm',
       'ct_dst_sport_ltm', 'attack_cat'],
      dtype='object')

`x_test` comes from `test_alldata_EDA.csv`, so it already has its column names. No renaming is needed here (`saved_dict['columns']` is only for raw files read with `header=None`).

In [280]:
# Creating new Feature
x_test['network_bytes'] = x_test['dbytes'] + x_test['sbytes']

In [281]:
# Droping all the unwanted columns (corr_col were already removed from test earlier, so skip missing ones)
dropable_col = saved_dict['to_drop'] + saved_dict['corr_col']
x_test.drop(columns=[c for c in dropable_col if c in x_test.columns], inplace=True)

In [282]:
x_test.shape

(15000, 35)

In [283]:
# Cleaning data using clean_data()
x_test = clean_data(x_test)

In [284]:
x_test.shape

(15000, 35)

In [285]:
# FE: applying log1p using apply_log1p()
x_test = apply_log1p(x_test)

In [286]:
x_test.shape

(15000, 35)

In [287]:
x_test.columns

Index(['proto', 'state', 'sttl', 'dttl', 'service', 'swin', 'trans_depth',
       'res_bdy_len', 'stime', 'sintpkt', 'dintpkt', 'tcprtt', 'ackdat',
       'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login',
       'ct_ftp_cmd', 'ct_srv_src', 'ct_dst_ltm', 'ct_src_ltm',
       'ct_dst_sport_ltm', 'dur_log1p', 'sbytes_log1p', 'dbytes_log1p',
       'sload_log1p', 'dload_log1p', 'spkts_log1p', 'stcpb_log1p',
       'dtcpb_log1p', 'smeansz_log1p', 'dmeansz_log1p', 'sjit_log1p',
       'djit_log1p', 'network_bytes_log1p'],
      dtype='object')

In [288]:
# Standardscaling using stanardize()
x_test = standardize(x_test)

In [289]:
x_test.head()

,proto,state,sttl,dttl,service,swin,trans_depth,res_bdy_len,stime,sintpkt,...,sload_log1p,dload_log1p,spkts_log1p,stcpb_log1p,dtcpb_log1p,smeansz_log1p,dmeansz_log1p,sjit_log1p,djit_log1p,network_bytes_log1p
0,tcp,FIN,-0.426176,-0.047006,None,0.836326,-0.293398,-0.085009,0.862570,-0.070252,...,-0.075531,0.484380,0.193810,0.759715,0.892420,0.054898,0.103942,0.323950,-0.508138,-0.001624
1,udp,CON,-0.426176,-0.047006,dns,-1.195706,-0.293398,-0.085009,0.858811,-0.070672,...,-0.096539,0.494787,-1.079531,-1.189628,-1.189259,-0.376919,0.094429,-0.995272,-0.899453,-0.974121
2,udp,INT,2.557956,-0.713187,dns,-1.195706,-0.293398,-0.085009,0.863037,-0.070670,...,1.308930,-1.883058,-1.079531,-1.189628,-1.189259,-0.765620,-1.853154,-0.995272,-0.899453,-1.416782
3,tcp,FIN,-0.426176,-0.047006,None,0.836326,-0.293398,-0.085009,-1.154352,-0.070494,...,0.054851,0.965509,0.970916,0.874611,0.874222,-0.633685,0.948840,0.079491,0.336689,1.023700
4,tcp,FIN,-0.426176,-0.047006,None,0.836326,-0.293398,-0.085009,0.849569,-0.070394,...,-0.097979,0.924621,1.263551,0.907663,0.809412,-0.711530,1.043277,0.221141,0.487921,1.275712


In [290]:
x_test['service'] = x_test['service'].replace('None', 'others')

for col, ohe in [('proto', ohe_proto), ('service', ohe_service), ('state', ohe_state)]:
    print(col, set(x_test[col].unique()) - set(ohe.categories_[0]))   # each set should be empty

x_test = ohencoding(x_test)

proto {'cpnx', 'pvp', 'idrp', 'ipv6-route', 'cbt', 'ipv6-frag', 'mhrp', 'ax.25', 'iplt', 'sm', 'tp++', 'hmp'}
service set()
state set()


In [291]:
# Onehot encoding categorical columns using ohencoding()
# x_test = ohencoding(x_test)

# x_train.filter(regex='^proto_').columns

# binary_cols = [c for c in x_train.columns if x_train[c].dropna().isin([0, 1]).all()]
# print(binary_cols)
saved_dict['binary_col']
x_train[saved_dict['binary_col']]

,is_sm_ips_ports,is_ftp_login
0,-0.042465,-0.133183
1,-0.042465,-0.133183
2,-0.042465,-0.133183
3,-0.042465,-0.133183
4,-0.042465,-0.133183
...,...,...
34995,-0.042465,-0.133183
34996,-0.042465,-0.133183
34997,-0.042465,-0.133183
34998,-0.042465,-0.133183


In [292]:
x_test.shape

(15000, 168)

In [293]:
# Final test data
x_test.head()

,sttl,dttl,swin,trans_depth,res_bdy_len,stime,sintpkt,dintpkt,tcprtt,ackdat,...,service_ssl,state_CLO,state_CON,state_ECO,state_FIN,state_INT,state_REQ,state_RST,state_URH,state_URN
0,-0.426176,-0.047006,0.836326,-0.293398,-0.085009,0.862570,-0.070252,-0.055906,-0.134821,-0.139255,...,0,0,0,0,1,0,0,0,0,0
1,-0.426176,-0.047006,-1.195706,-0.293398,-0.085009,0.858811,-0.070672,-0.056630,-0.148237,-0.144872,...,0,0,1,0,0,0,0,0,0,0
2,2.557956,-0.713187,-1.195706,-0.293398,-0.085009,0.863037,-0.070670,-0.056635,-0.148237,-0.144872,...,0,0,0,0,0,1,0,0,0,0
3,-0.426176,-0.047006,0.836326,-0.293398,-0.085009,-1.154352,-0.070494,-0.056299,-0.130029,-0.137940,...,0,0,0,0,1,0,0,0,0,0
4,-0.426176,-0.047006,0.836326,-0.293398,-0.085009,0.849569,-0.070394,-0.056099,-0.133096,-0.139002,...,0,0,0,0,1,0,0,0,0,0


In [294]:
# Matching test data columns with train data columns
# all(x_train.columns == x_test.columns)

train.columns,train.shape

(Index(['proto', 'state', 'sttl', 'dttl', 'service', 'swin', 'trans_depth',
        'res_bdy_len', 'stime', 'sintpkt', 'dintpkt', 'tcprtt', 'ackdat',
        'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login',
        'ct_ftp_cmd', 'ct_srv_src', 'ct_dst_ltm', 'ct_src_ltm',
        'ct_dst_sport_ltm', 'label', 'dur_log1p', 'sbytes_log1p',
        'dbytes_log1p', 'sload_log1p', 'dload_log1p', 'spkts_log1p',
        'stcpb_log1p', 'dtcpb_log1p', 'smeansz_log1p', 'dmeansz_log1p',
        'sjit_log1p', 'djit_log1p', 'network_bytes_log1p'],
       dtype='object'),
 (35000, 36))

In [295]:
# Save the exact model inputs as sparse matrices plus feature/label metadata.
assert x_train.columns.equals(x_test.columns), "Train/test feature columns do not match."
assert len(x_train) == len(y_train), "Training features and labels have different row counts."
assert len(x_test) == len(y_test), "Test features and labels have different row counts."
assert all(pd.api.types.is_numeric_dtype(dtype) for dtype in x_train.dtypes), "Model features must all be numeric."

from scipy.sparse import csr_matrix, hstack, save_npz

def to_csr(features):
    columns = list(features.columns)
    dense_columns = [column for column in columns if not isinstance(features[column].dtype, pd.SparseDtype)]
    sparse_columns = [column for column in columns if isinstance(features[column].dtype, pd.SparseDtype)]
    assert columns == dense_columns + sparse_columns, "Dense and one-hot columns are not in the expected order."
    dense = csr_matrix(features[dense_columns].to_numpy(dtype=np.float64, copy=False))
    encoded = features[sparse_columns].sparse.to_coo().tocsr()
    return hstack((dense, encoded), format='csr', dtype=np.float64), columns

os.makedirs(file_path, exist_ok=True)
for split_name, features, target in [
    ('final_train', x_train, y_train),
    ('final_test', x_test, y_test),
]:
    matrix, columns = to_csr(features)
    matrix_path = os.path.join(file_path, split_name + '.npz')
    matrix_temp_path = os.path.join(file_path, split_name + '.tmp.npz')
    metadata_path = os.path.join(file_path, split_name + '_meta.pkl')
    metadata_temp_path = metadata_path + '.tmp'
    save_npz(matrix_temp_path, matrix, compressed=True)
    with open(metadata_temp_path, 'wb') as output_file:
        pickle.dump((columns, target), output_file, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(matrix_temp_path, matrix_path)
    os.replace(metadata_temp_path, metadata_path)
    print(f"Saved {matrix_path}: features={matrix.shape}, labels={target.shape}")
    del matrix, columns

Saved .final_ipynb/final_train.npz: features=(35000, 168), labels=(35000,)
Saved .final_ipynb/final_test.npz: features=(15000, 168), labels=(15000,)
